# XDMA DCIM HBM Bring-up Test

本 notebook 对应 `xdma_dcim_hbm` 当前 Tcl 配置，覆盖三类检查：
1. `XDMA <-> HBM` 读写一致性
2. `CDMA <-> PE ibuf/obuf` staging 路径
3. 一条最小 `HBM -> ibuf -> PE/DCIM -> obuf -> HBM` 闭环

当前地址映射来自 `scripts/ip/bd/xdma_dcim_hbm/address.tcl`：
- **仅 SAXI_00** 参与主机地址映射：低 4GB 连续映射全部伪通道（Vivado HBM 要求段偏移与物理固定地址对齐，不能把 SAXI_01..15 挪到任意高端地址另行映射）。
- 伪通道 `k` 对应主机偏移 **`k * 256MB`**（仍在低 4GB 内）。
- `PE ibuf`：`0x100000000`
- `PE obuf`：`0x100010000`
- `CDMA regs`：`0x100020000`
- `PE cfg0..3`：`0x100030000`..`0x100033000`
- `PE ctrl/status`：`0x100034000` / `0x100035000`

默认 `HBM_PORT = 0`。切换伪通道时改为 `HBM_PORT = k`，基地址为 **`k * 256MB`**（仍在低 4GB）。

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import hashlib
import subprocess
import time

import numpy as np

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (candidate / "tests" / "bin" / "xdma_rw.exe").exists() or (candidate / "tests" / "bin" / "xdma_rw").exists():
        REPO_ROOT = candidate
        break

TEST_DIR = REPO_ROOT / "tests" / "xdma_dcim_hbm"
if (NOTEBOOK_DIR / "rw_test.ipynb").exists() and NOTEBOOK_DIR.name == "xdma_dcim_hbm":
    TEST_DIR = NOTEBOOK_DIR
TEST_DIR.mkdir(parents=True, exist_ok=True)

BIN_DIR = REPO_ROOT / "tests" / "bin"
for candidate in [BIN_DIR / "xdma_rw.exe", BIN_DIR / "xdma_rw", NOTEBOOK_DIR / "bin" / "xdma_rw.exe", NOTEBOOK_DIR / "bin" / "xdma_rw"]:
    if candidate.exists():
        XDMA_EXE = candidate
        break
else:
    XDMA_EXE = BIN_DIR / "xdma_rw.exe"

for candidate in [BIN_DIR / "xdma_info.exe", BIN_DIR / "xdma_info", NOTEBOOK_DIR / "bin" / "xdma_info.exe", NOTEBOOK_DIR / "bin" / "xdma_info"]:
    if candidate.exists():
        XDMA_INFO_EXE = candidate
        break
else:
    XDMA_INFO_EXE = None

DATA_DIR = TEST_DIR / "data"
READBACK_DIR = TEST_DIR / "readbacks"
REG_DIR = TEST_DIR / "reg_io"
for path in (DATA_DIR, READBACK_DIR, REG_DIR):
    path.mkdir(exist_ok=True)

CHANNEL = 0
HBM_PORT = 0
CMD_TIMEOUT_SEC = 30
HBM_SEGMENT_SIZE = 0x10000000
HBM_BASE = HBM_PORT * HBM_SEGMENT_SIZE
BRAM_WINDOW_BYTES = 64 * 1024
BRAM_WORD_BYTES = 32
MODE_INT8 = 0b110

# Address map (must match scripts/ip/bd/xdma_dcim_hbm/address.tcl)
# Local PE/CDMA/GPIO aperture starts after the 4 GB HBM window at 0x100000000
IBUF_BASE = 0x100000000
OBUF_BASE = 0x100010000
CDMA_BASE = 0x100020000
PE_CFG0_BASE = 0x100030000
PE_CFG1_BASE = 0x100031000
PE_CFG2_BASE = 0x100032000
PE_CFG3_BASE = 0x100033000
PE_CTRL_BASE = 0x100034000
PE_STATUS_BASE = 0x100035000

assert XDMA_EXE.exists(), f"Cannot find XDMA tool: {XDMA_EXE}"
print(f"XDMA tool : {XDMA_EXE}")
print(f"Test dir  : {TEST_DIR}")
print(f"HBM port  : {HBM_PORT}")
print(f"HBM base  : 0x{HBM_BASE:X}")
print(f"HBM span  : 0x{HBM_BASE:X}..0x{HBM_BASE + HBM_SEGMENT_SIZE:X}")
print(f"PE ibuf   : 0x{IBUF_BASE:X}..0x{IBUF_BASE + BRAM_WINDOW_BYTES:X}")
print(f"PE obuf   : 0x{OBUF_BASE:X}..0x{OBUF_BASE + BRAM_WINDOW_BYTES:X}")
print(f"CDMA regs : 0x{CDMA_BASE:X}..0x{CDMA_BASE + 0x10000:X}")
print(f"Channel   : h2c_{CHANNEL}/c2h_{CHANNEL}")
if XDMA_INFO_EXE is not None:
    print(f"XDMA info : {XDMA_INFO_EXE}")


In [ ]:
@dataclass
class CmdResult:
    cmd: list[str]
    returncode: int
    stdout: str
    stderr: str
    elapsed_sec: float

    @property
    def ok(self) -> bool:
        return self.returncode == 0


def _subprocess_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    return str(value)


def run_cmd(cmd: list[str], timeout: int = CMD_TIMEOUT_SEC) -> CmdResult:
    start = time.perf_counter()
    try:
        cp = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=timeout)
        return CmdResult(cmd, cp.returncode, cp.stdout, cp.stderr, time.perf_counter() - start)
    except subprocess.TimeoutExpired as exc:
        elapsed = time.perf_counter() - start
        stderr = _subprocess_text(exc.stderr)
        if stderr:
            stderr += "\n"
        stderr += f"TIMEOUT: command did not finish within {timeout}s"
        return CmdResult(cmd, -124, _subprocess_text(exc.stdout), stderr, elapsed)


def print_result(name: str, result: CmdResult) -> None:
    print(f"{name}: {'PASS' if result.ok else 'FAIL'}, ret={result.returncode}, elapsed={result.elapsed_sec:.3f}s")
    print("CMD:", " ".join(result.cmd))
    if result.stdout.strip():
        print("STDOUT:")
        print(result.stdout.strip())
    if result.stderr.strip():
        print("STDERR:")
        print(result.stderr.strip())


def make_pattern(size: int, seed: int = 0) -> bytes:
    value = seed & 0xFFFFFFFF
    out = bytearray()
    for _ in range(size):
        value = (1664525 * value + 1013904223) & 0xFFFFFFFF
        out.append((value >> 24) & 0xFF)
    return bytes(out)


def validate_hbm_range(offset: int, size: int) -> None:
    assert offset >= 0
    assert size > 0
    assert offset + size <= HBM_SEGMENT_SIZE, f"range exceeds HBM segment: off=0x{offset:X}, size=0x{size:X}"


def assert_local_range(offset: int, size: int) -> None:
    assert offset >= 0
    assert size > 0
    assert offset + size <= BRAM_WINDOW_BYTES, f"range exceeds local 64KB window: off=0x{offset:X}, size=0x{size:X}"


def xdma_write(addr: int, payload_path: Path, timeout: int = CMD_TIMEOUT_SEC) -> CmdResult:
    return run_cmd([str(XDMA_EXE), f"h2c_{CHANNEL}", "write", hex(addr), "-b", "-f", str(payload_path)], timeout=timeout)


def xdma_read(addr: int, size: int, out_path: Path, timeout: int = CMD_TIMEOUT_SEC) -> CmdResult:
    if out_path.exists():
        out_path.unlink()
    return run_cmd([str(XDMA_EXE), f"c2h_{CHANNEL}", "read", hex(addr), "-l", hex(size), "-b", "-f", str(out_path)], timeout=timeout)


def mmio_write_u32(addr: int, value: int, label: str = "") -> CmdResult:
    name = label or f"wr_{addr:X}"
    path = REG_DIR / f"{name}.bin"
    path.write_bytes((value & 0xFFFFFFFF).to_bytes(4, "little"))
    return xdma_write(addr, path)


def mmio_read_u32(addr: int, label: str = "") -> tuple[int, CmdResult]:
    name = label or f"rd_{addr:X}"
    path = REG_DIR / f"{name}.bin"
    result = xdma_read(addr, 4, path)
    value = int.from_bytes(path.read_bytes()[:4], "little") if result.ok and path.exists() else 0
    return value, result


def compare_bytes(expected: bytes, actual_path: Path, max_diff: int = 16) -> tuple[bool, str]:
    if not actual_path.exists():
        return False, f"missing readback file: {actual_path}"
    actual = actual_path.read_bytes()
    if expected == actual:
        digest = hashlib.sha256(actual).hexdigest()
        return True, f"match: {len(actual)} bytes, sha256={digest}"
    lines = [f"mismatch: expected_len={len(expected)}, actual_len={len(actual)}"]
    for index, (a, b) in enumerate(zip(expected, actual)):
        if a != b:
            lines.append(f"0x{index:08X}: expected=0x{a:02X}, actual=0x{b:02X}")
            if len(lines) > max_diff:
                break
    return False, "\n".join(lines)


def run_xdma_info(timeout: int = 10) -> CmdResult | None:
    if XDMA_INFO_EXE is None:
        print("xdma_info.exe not found")
        return None
    result = run_cmd([str(XDMA_INFO_EXE)], timeout=timeout)
    print_result("XDMA INFO", result)
    return result


def probe_rw_abs(base_addr: int, size: int, label: str, seed: int = 0x5A) -> dict:
    payload = make_pattern(size, seed=seed)
    write_path = DATA_DIR / f"{label}_write.bin"
    read_path = READBACK_DIR / f"{label}_read.bin"
    write_path.write_bytes(payload)

    wr = xdma_write(base_addr, write_path, timeout=10)
    print_result(f"PROBE WRITE {label}", wr)
    assert wr.ok, f"probe write failed: {label}"

    rd = xdma_read(base_addr, size, read_path, timeout=10)
    print_result(f"PROBE READ {label}", rd)
    assert rd.ok, f"probe read failed: {label}"

    match, detail = compare_bytes(payload, read_path)
    print(f"PROBE COMPARE {label}:", "PASS" if match else "FAIL")
    print(detail)
    assert match, f"probe compare failed: {label}"
    return {"label": label, "addr": base_addr, "size": size, "detail": detail}


In [ ]:
def run_hbm_smoke() -> list[dict]:
    cases = [
        (0x00000000, 0x00000100, 0x00000011),
        (0x00001000, 0x00001000, 0x00000022),
        (0x00010040, 0x00000158, 0x00001234),
        (0x0FFF0000, 0x00002000, 0x0000ABCD),
    ]
    results = []
    for offset, size, seed in cases:
        validate_hbm_range(offset, size)
        addr = HBM_BASE + offset
        payload = make_pattern(size, seed=seed)
        write_path = DATA_DIR / f"hbm_write_p{HBM_PORT:02d}_off{offset:08X}_size{size:08X}_seed{seed:08X}.bin"
        read_path = READBACK_DIR / f"hbm_read_p{HBM_PORT:02d}_off{offset:08X}_size{size:08X}_seed{seed:08X}.bin"
        write_path.write_bytes(payload)
        print(f"\n=== HBM CASE offset=0x{offset:08X} size=0x{size:08X} addr=0x{addr:X} ===")
        wr = xdma_write(addr, write_path)
        print_result("HBM WRITE", wr)
        assert wr.ok
        rd = xdma_read(addr, size, read_path)
        print_result("HBM READ", rd)
        assert rd.ok
        match, detail = compare_bytes(payload, read_path)
        print("HBM COMPARE:", "PASS" if match else "FAIL")
        print(detail)
        assert match
        results.append({"offset": offset, "size": size, "seed": seed, "detail": detail})
    return results


def run_local_window_smoke() -> list[dict]:
    probes = [
        (IBUF_BASE + 0x0000, 0x100, "ibuf_direct", 0x1B0F),
        (OBUF_BASE + 0x0200, 0x100, "obuf_direct", 0x2B0F),
    ]
    results = []
    for addr, size, label, seed in probes:
        assert_local_range(addr & 0xFFFF, size)
        results.append(probe_rw_abs(addr, size, label, seed=seed))
    return results


In [ ]:
CDMA_CR = 0x00
CDMA_SR = 0x04
CDMA_SA = 0x18
CDMA_SA_MSB = 0x1C
CDMA_DA = 0x20
CDMA_DA_MSB = 0x24
CDMA_BTT = 0x28

CDMA_SR_IDLE = 1 << 1
CDMA_SR_ERR_MASK = 0x00000770


def cdma_status() -> int:
    value, result = mmio_read_u32(CDMA_BASE + CDMA_SR, "cdma_sr")
    assert result.ok, "CDMA status read failed"
    return value


def cdma_wait_idle(timeout_sec: float = 2.0) -> int:
    deadline = time.time() + timeout_sec
    last = 0
    while time.time() < deadline:
        last = cdma_status()
        if last & CDMA_SR_ERR_MASK:
            raise RuntimeError(f"CDMA error status: 0x{last:08X}")
        if last & CDMA_SR_IDLE:
            return last
        time.sleep(0.001)
    raise TimeoutError(f"CDMA did not become idle, last status=0x{last:08X}")


def cdma_reset() -> None:
    result = mmio_write_u32(CDMA_BASE + CDMA_CR, 1 << 2, "cdma_reset")
    print_result("CDMA RESET", result)
    assert result.ok
    time.sleep(0.01)
    cdma_wait_idle(timeout_sec=2.0)


def cdma_write_addr(reg_lo: int, reg_hi: int, value: int, label: str) -> None:
    lo = value & 0xFFFFFFFF
    hi = (value >> 32) & 0xFFFFFFFF
    result_lo = mmio_write_u32(CDMA_BASE + reg_lo, lo, f"{label}_lo")
    print_result(f"CDMA WRITE {label}_LO", result_lo)
    assert result_lo.ok
    result_hi = mmio_write_u32(CDMA_BASE + reg_hi, hi, f"{label}_hi")
    print_result(f"CDMA WRITE {label}_HI", result_hi)
    assert result_hi.ok


def cdma_memcpy(src_addr: int, dst_addr: int, size: int, label: str = "cdma") -> None:
    assert size > 0
    assert src_addr % 4 == 0 and dst_addr % 4 == 0
    cdma_wait_idle(timeout_sec=2.0)
    cdma_write_addr(CDMA_SA, CDMA_SA_MSB, src_addr, f"{label}_sa")
    cdma_write_addr(CDMA_DA, CDMA_DA_MSB, dst_addr, f"{label}_da")
    result = mmio_write_u32(CDMA_BASE + CDMA_BTT, size, f"{label}_btt")
    print_result("CDMA WRITE BTT", result)
    assert result.ok
    status = cdma_wait_idle(timeout_sec=5.0)
    print(f"CDMA {label} complete, status=0x{status:08X}")


def run_cdma_staging_smoke(size: int = 0x200) -> dict:
    assert_local_range(0x0000, size)
    assert_local_range(0x0400, size)
    validate_hbm_range(0x8000, size)
    validate_hbm_range(0xC000, size)

    hbm_src = HBM_BASE + 0x8000
    ibuf_dst = IBUF_BASE + 0x0000
    obuf_src = OBUF_BASE + 0x0400
    hbm_dst = HBM_BASE + 0xC000

    src_payload = make_pattern(size, seed=0x5147)
    src_path = DATA_DIR / "cdma_hbm_src.bin"
    src_path.write_bytes(src_payload)
    wr = xdma_write(hbm_src, src_path)
    print_result("XDMA WRITE HBM SRC", wr)
    assert wr.ok

    obuf_payload = make_pattern(size, seed=0xA55A)
    obuf_path = DATA_DIR / "cdma_obuf_src.bin"
    obuf_path.write_bytes(obuf_payload)
    wr = xdma_write(obuf_src, obuf_path)
    print_result("XDMA WRITE OBUF SRC", wr)
    assert wr.ok

    cdma_reset()
    cdma_memcpy(hbm_src, ibuf_dst, size, "hbm_to_ibuf")
    ibuf_readback = READBACK_DIR / "cdma_ibuf_read.bin"
    rd = xdma_read(ibuf_dst, size, ibuf_readback)
    print_result("XDMA READ IBUF", rd)
    assert rd.ok
    match_ibuf, detail_ibuf = compare_bytes(src_payload, ibuf_readback)
    print("IBUF COMPARE:", "PASS" if match_ibuf else "FAIL")
    print(detail_ibuf)
    assert match_ibuf

    cdma_memcpy(obuf_src, hbm_dst, size, "obuf_to_hbm")
    hbm_readback = READBACK_DIR / "cdma_hbm_read.bin"
    rd = xdma_read(hbm_dst, size, hbm_readback)
    print_result("XDMA READ HBM DST", rd)
    assert rd.ok
    match_hbm, detail_hbm = compare_bytes(obuf_payload, hbm_readback)
    print("HBM DST COMPARE:", "PASS" if match_hbm else "FAIL")
    print(detail_hbm)
    assert match_hbm

    return {
        "ibuf": detail_ibuf,
        "hbm": detail_hbm,
        "size": size,
    }


In [ ]:
def pack_activation_rows(act: np.ndarray) -> bytes:
    assert act.dtype == np.int8 and act.ndim == 2 and act.shape[1] == 16
    payload = bytearray()
    for row in range(0, act.shape[0], 2):
        lo = act[row].astype(np.uint8).tobytes()
        hi = act[row + 1].astype(np.uint8).tobytes() if row + 1 < act.shape[0] else bytes(16)
        payload += lo + hi
    return bytes(payload)


def pack_weight_tile(weight: np.ndarray) -> bytes:
    assert weight.dtype == np.int8 and weight.shape == (16, 8)
    return weight.astype(np.uint8).tobytes(order="C")


def golden_int8_matmul(act: np.ndarray, weight: np.ndarray, acc: int = 0) -> np.ndarray:
    assert act.dtype == np.int8 and weight.dtype == np.int8
    out = act.astype(np.int32) @ weight.astype(np.int32)
    if acc in (0, 1):
        return out.astype(np.int32)
    assert acc in (2, 4, 8, 16), "RTL accepts only acc=0/1/2/4/8/16"
    assert out.shape[0] % acc == 0, "raw_rows must be divisible by acc"
    return out.reshape(out.shape[0] // acc, acc, out.shape[1]).sum(axis=1).astype(np.int32)


def pack_expected_rows(rows: np.ndarray) -> bytes:
    assert rows.dtype == np.int32 and rows.ndim == 2 and rows.shape[1] == 8
    return rows.astype("<i4", copy=False).tobytes(order="C")


def decode_result_rows(payload: bytes) -> np.ndarray:
    assert len(payload) % 32 == 0
    return np.frombuffer(payload, dtype="<i4").reshape(-1, 8)


def make_case(raw_rows: int = 16, acc: int = 0, seed: int = 1234) -> dict:
    assert raw_rows > 0
    rng = np.random.default_rng(seed)
    act = rng.integers(-128, 128, size=(raw_rows, 16), dtype=np.int8)
    weight = rng.integers(-128, 128, size=(16, 8), dtype=np.int8)
    return make_case_from_arrays(act, weight, acc=acc)


def make_case_from_arrays(act: np.ndarray, weight: np.ndarray, acc: int = 0) -> dict:
    assert act.dtype == np.int8 and act.ndim == 2 and act.shape[1] == 16
    assert weight.dtype == np.int8 and weight.shape == (16, 8)
    expected = golden_int8_matmul(act, weight, acc=acc)
    return {
        "raw_rows": act.shape[0],
        "acc": acc,
        "act": act,
        "weight": weight,
        "weight_payload": pack_weight_tile(weight),
        "act_payload": pack_activation_rows(act),
        "expected_rows": expected,
        "expected_payload": pack_expected_rows(expected),
    }


def make_signed_extreme_case(raw_rows: int = 16, acc: int = 2) -> dict:
    assert raw_rows > 0
    values = np.array([-128, -127, -64, -3, -1, 0, 1, 2, 3, 7, 15, 31, 63, 64, 126, 127], dtype=np.int8)
    act = np.vstack([np.roll(values, row) for row in range(raw_rows)]).astype(np.int8)
    weight = np.vstack([np.roll(values[:8], col) for col in range(16)]).astype(np.int8)
    return make_case_from_arrays(act, weight, acc=acc)


def summarize_case(case: dict, name: str) -> None:
    print(
        f"{name}: rows={case['raw_rows']}, acc={case['acc']}, "
        f"expected_rows={case['expected_rows'].shape[0]}, "
        f"payloads={len(case['weight_payload'])}/{len(case['act_payload'])}/{len(case['expected_payload'])} bytes"
    )


def pe_status() -> dict:
    status, status_result = mmio_read_u32(PE_STATUS_BASE + 0x00, "pe_status")
    error_code, err_result = mmio_read_u32(PE_STATUS_BASE + 0x08, "pe_error_code")
    assert status_result.ok and err_result.ok, "PE status read failed"
    return {
        "raw": status,
        "busy": bool(status & 0x1),
        "done": bool(status & 0x2),
        "error": bool(status & 0x4),
        "config_ready": bool(status & 0x8),
        "error_code": error_code,
    }


def pe_ctrl_value(acc: int, start: bool = False, config_valid: bool = True) -> int:
    return ((1 if start else 0) << 0) | ((1 if config_valid else 0) << 1) | (MODE_INT8 << 2) | ((acc & 0x1F) << 5)


def pe_clear_done_if_needed() -> None:
    status = pe_status()
    if status["done"] and not status["busy"]:
        result = mmio_write_u32(PE_CTRL_BASE, 0x1, "pe_clear_done")
        print_result("PE CLEAR DONE", result)
        assert result.ok
        time.sleep(0.002)
        result = mmio_write_u32(PE_CTRL_BASE, 0x0, "pe_clear_done_low")
        assert result.ok


def pe_start(wsrc_base: int, asrc_base: int, dst_base: int, raw_rows: int, acc: int) -> dict:
    assert wsrc_base % BRAM_WORD_BYTES == 0
    assert asrc_base % BRAM_WORD_BYTES == 0
    assert dst_base % BRAM_WORD_BYTES == 0
    pe_clear_done_if_needed()
    for name, base, value in [
        ("wsrc", PE_CFG0_BASE, wsrc_base),
        ("asrc", PE_CFG1_BASE, asrc_base),
        ("dst", PE_CFG2_BASE, dst_base),
        ("raw_rows", PE_CFG3_BASE, raw_rows),
    ]:
        result = mmio_write_u32(base, value, f"pe_cfg_{name}")
        print_result(f"PE CFG {name}", result)
        assert result.ok

    ctrl = pe_ctrl_value(acc, start=False, config_valid=True)
    result = mmio_write_u32(PE_CTRL_BASE, ctrl, "pe_ctrl_cfg")
    print_result("PE CTRL config_valid", result)
    assert result.ok
    time.sleep(0.002)

    result = mmio_write_u32(PE_CTRL_BASE, pe_ctrl_value(acc, start=True, config_valid=True), "pe_ctrl_start")
    print_result("PE CTRL start", result)
    assert result.ok
    time.sleep(0.002)
    result = mmio_write_u32(PE_CTRL_BASE, ctrl, "pe_ctrl_start_low")
    assert result.ok

    deadline = time.time() + 5.0
    last = {}
    while time.time() < deadline:
        last = pe_status()
        if last["error"]:
            raise RuntimeError(f"PE error: {last}")
        if last["done"]:
            return last
        time.sleep(0.001)
    raise TimeoutError(f"PE did not complete, last status={last}")


def run_dcim_case(case: dict, name: str = "case0") -> dict:
    w_hbm_off = 0x0000
    a_hbm_off = 0x1000
    r_hbm_off = 0x4000
    w_ibuf_off = 0x0000
    a_ibuf_off = 0x0100
    r_obuf_off = 0x0000

    weight_payload = case["weight_payload"]
    act_payload = case["act_payload"]
    expected_payload = case["expected_payload"]

    validate_hbm_range(w_hbm_off, len(weight_payload))
    validate_hbm_range(a_hbm_off, len(act_payload))
    validate_hbm_range(r_hbm_off, len(expected_payload))
    assert_local_range(w_ibuf_off, len(weight_payload))
    assert_local_range(a_ibuf_off, len(act_payload))
    assert_local_range(r_obuf_off, len(expected_payload))

    weight_path = DATA_DIR / f"{name}_weight.bin"
    act_path = DATA_DIR / f"{name}_act.bin"
    out_path = READBACK_DIR / f"{name}_result.bin"
    weight_path.write_bytes(weight_payload)
    act_path.write_bytes(act_payload)

    wr_w = xdma_write(HBM_BASE + w_hbm_off, weight_path)
    print_result("XDMA WRITE HBM weight", wr_w)
    assert wr_w.ok
    wr_a = xdma_write(HBM_BASE + a_hbm_off, act_path)
    print_result("XDMA WRITE HBM activation", wr_a)
    assert wr_a.ok

    cdma_reset()
    cdma_memcpy(HBM_BASE + w_hbm_off, IBUF_BASE + w_ibuf_off, len(weight_payload), f"{name}_w_to_ibuf")
    cdma_memcpy(HBM_BASE + a_hbm_off, IBUF_BASE + a_ibuf_off, len(act_payload), f"{name}_a_to_ibuf")

    status = pe_start(
        wsrc_base=w_ibuf_off,
        asrc_base=a_ibuf_off,
        dst_base=r_obuf_off,
        raw_rows=case["raw_rows"],
        acc=case["acc"],
    )
    print("PE DONE:", status)

    cdma_memcpy(OBUF_BASE + r_obuf_off, HBM_BASE + r_hbm_off, len(expected_payload), f"{name}_obuf_to_hbm")
    rd = xdma_read(HBM_BASE + r_hbm_off, len(expected_payload), out_path)
    print_result("XDMA READ HBM result", rd)
    assert rd.ok

    match, detail = compare_bytes(expected_payload, out_path)
    print("COMPARE:", "PASS" if match else "FAIL")
    print(detail)
    if out_path.exists():
        print("hardware rows:")
        print(decode_result_rows(out_path.read_bytes()))
        print("expected rows:")
        print(case["expected_rows"])
    assert match, "DCIM result mismatch"
    return {"pass": match, "status": status, "readback": out_path}


## Bring-up Order

建议按下面顺序执行：
1. `run_xdma_info()`
2. `run_hbm_smoke()`
3. `run_local_window_smoke()`
4. `run_cdma_staging_smoke()`
5. 单个 `run_dcim_case(...)`
6. 小回归 `regression_cases`

如果第 2 步失败，先查 XDMA/HBM；如果第 3 步失败，查 local aperture 和 BRAM controller；如果第 4 步失败，查 CDMA 64-bit 地址和互连；如果前 4 步都过但第 5 步失败，再查 PE/DCIM 本身。

In [ ]:
run_xdma_info()

In [ ]:
hbm_smoke = run_hbm_smoke()
hbm_smoke

In [ ]:
local_smoke = run_local_window_smoke()
local_smoke

In [ ]:
cdma_smoke = run_cdma_staging_smoke()
cdma_smoke

In [ ]:
case = make_case(raw_rows=16, acc=0, seed=0xDC10)
summarize_case(case, "int8_rows16_acc0")
result = run_dcim_case(case, name="int8_rows16_acc0")
result

In [ ]:
regression_cases = [
    ("int8_rows16_acc1_random", make_case(raw_rows=16, acc=1, seed=0xDC11)),
    ("int8_rows16_acc4_random", make_case(raw_rows=16, acc=4, seed=0xDC12)),
    ("int8_rows16_acc2_extremes", make_signed_extreme_case(raw_rows=16, acc=2)),
]

regression_results = []
for name, case in regression_cases:
    summarize_case(case, name)
    regression_results.append((name, run_dcim_case(case, name=name)))

regression_results